In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FILE = Path("dataset/ecommerce_sales_mysql_import.csv")
df = pd.read_csv(DATA_FILE)
df.head()


,customer_id,gender,region,age,age_cleaned,age_group,age_missing_flag,age_model,age_treatment_method,product_name,category,unit_price,quantity,total_price,shipping_fee,shipping_status,order_date,year,month,year_month
0,CUST0268,Male,North,NaN,21.0,18-24,1,21,Recovered from same Customer ID,Monitor,Electronics,300.0,5,1500.0,13.31,Returned,2023-12-08,2023,12,2023-12
1,CUST0046,Male,West,22.0,22.0,18-24,0,22,Original,Headphones,Accessories,100.0,2,200.0,6.93,In Transit,2023-04-09,2023,4,2023-04
2,CUST0169,Female,South,54.0,54.0,45-54,0,54,Original,Monitor,Electronics,300.0,1,300.0,11.31,Returned,2023-08-28,2023,8,2023-08
3,CUST0002,Male,North,23.0,23.0,18-24,0,23,Original,Headphones,Accessories,100.0,5,500.0,12.22,Delivered,2023-01-18,2023,1,2023-01
4,CUST0173,Female,South,NaN,65.0,65+,1,65,Recovered from same Customer ID,Laptop,Electronics,1500.0,3,4500.0,5.40,Delivered,2023-01-19,2023,1,2023-01


The cleaned e-commerce sales dataset was loaded using pandas. Each row represents one transaction or order record. The goal of this phase is to inspect the dataset structure, check data quality, analyze missing values, and prepare the data for deeper exploratory analysis.

In [28]:
print("shape:", df.shape) ## Number of rows and columns
print("INFO: ")  ## columns,datatypes, and non-null counts
df.info()
print("dTypes:") ## exact data type per column
df.dtypes
print("Describe: ") # Summary Statistics for numeric fields
df.describe()
df.describe(include="object") ## Summary of Categorical/text fields
df.head() ## first
df.tail() ## and last row 
## Checks if data looks correct

shape: (1000, 20)
INFO: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           1000 non-null   object        
 1   gender                1000 non-null   object        
 2   region                1000 non-null   object        
 3   age                   900 non-null    float64       
 4   age_cleaned           993 non-null    float64       
 5   age_group             1000 non-null   object        
 6   age_missing_flag      1000 non-null   int64         
 7   age_model             1000 non-null   int64         
 8   age_treatment_method  1000 non-null   object        
 9   product_name          1000 non-null   object        
 10  category              1000 non-null   object        
 11  unit_price            1000 non-null   float64       
 12  quantity              1000 non-null   int64         

,customer_id,gender,region,age,age_cleaned,age_group,age_missing_flag,age_model,age_treatment_method,product_name,category,unit_price,quantity,total_price,shipping_fee,shipping_status,order_date,year,month,year_month
995,CUST0201,Female,South,49.0,49.0,45-54,0,49,Original,Headphones,Accessories,100.0,1,100.0,17.21,In Transit,2023-01-05,2023,1,2023-01
996,CUST0133,Male,East,47.0,47.0,45-54,0,47,Original,Laptop,Electronics,1500.0,1,1500.0,19.19,Delivered,2023-04-01,2023,4,2023-04
997,CUST0055,Female,North,NaN,45.0,45-54,1,45,Recovered from same Customer ID,Mouse,Accessories,30.0,5,150.0,19.35,Delivered,2023-10-20,2023,10,2023-10
998,CUST0023,Female,South,29.0,29.0,25-34,0,29,Original,Laptop,Electronics,1500.0,5,7500.0,10.36,Returned,2023-01-07,2023,1,2023-01
999,CUST0040,Male,Unknown,29.0,29.0,25-34,0,29,Original,Smartwatch,Wearables,200.0,1,200.0,7.53,Delivered,2023-01-30,2023,1,2023-01


The dataset contains 1,000 rows and 20 columns. The columns include customer information, product details, transaction values, shipping information, date fields, and age-treatment fields. This confirms that the dataset contains both numeric and categorical variables needed for descriptive statistics, segmentation, trend analysis, and basic predictive analysis.

In [11]:
## Validate data type
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

print("Earliest order date:", df["order_date"].min())
print("Latest order date:", df["order_date"].max())
print("Invalid order dates:", df["order_date"].isna().sum())

Earliest order date: 2023-01-01 00:00:00
Latest order date: 2024-01-03 00:00:00
Invalid order dates: 0


The order_date column was converted into datetime format so it can be used for time-based analysis. The dataset covers transactions from January 1, 2023 to January 3, 2024. Any invalid date values were checked using missing-value detection after conversion.

In [18]:
## Missing Value Analysis
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

## missing_table (uncomment for eeverything)
missing_table[missing_table["missing_count"] > 0] ## only shows columns with missing values

## next is unknown values seperately, because uknown is not technically missing

,missing_count,missing_percentage
age,100,10.0
age_cleaned,7,0.7


In [19]:
## Missing Value Analysis
print("UNKNOWN AREA: \n")
unknown_cols = ["region", "shipping_status", "age_group"]

for col in unknown_cols:
    print(f"\n{col.upper()}")
    print(df[col].value_counts(dropna=False))

UNKNOWN AREA: 


REGION
region
West       246
South      244
East       231
North      229
Unknown     50
Name: count, dtype: int64

SHIPPING_STATUS
shipping_status
In Transit    329
Delivered     313
Returned      308
Unknown        50
Name: count, dtype: int64

AGE_GROUP
age_group
55-64      214
45-54      200
35-44      190
25-34      150
65+        146
18-24       93
Unknown      7
Name: count, dtype: int64


This is important because Unknown values are intentional categories, not blank values

In [20]:
## AGE TREATMENT CHECK
age_cols = [
    "age",
    "age_cleaned",
    "age_group",
    "age_missing_flag",
    "age_model",
    "age_treatment_method"
]

df[age_cols].head(15)

,age,age_cleaned,age_group,age_missing_flag,age_model,age_treatment_method
0,NaN,21.0,18-24,1,21,Recovered from same Customer ID
1,22.0,22.0,18-24,0,22,Original
2,54.0,54.0,45-54,0,54,Original
3,23.0,23.0,18-24,0,23,Original
4,NaN,65.0,65+,1,65,Recovered from same Customer ID
5,64.0,64.0,55-64,0,64,Original
6,27.0,27.0,25-34,0,27,Original
7,18.0,18.0,18-24,0,18,Original
8,22.0,22.0,18-24,0,22,Original
9,29.0,29.0,25-34,0,29,Original


In [25]:
df[age_cols].isna().sum()


age                     100
age_cleaned               7
age_group                 0
age_missing_flag          0
age_model                 0
age_treatment_method      0
dtype: int64

In [22]:
df["age_treatment_method"].value_counts(dropna=False)

age_treatment_method
Original                                                      900
Recovered from same Customer ID                                93
Unknown for age analysis; median-imputed for modeling only      7
Name: count, dtype: int64

In [23]:
df["age_treatment_method"].value_counts(dropna=False)

age_treatment_method
Original                                                      900
Recovered from same Customer ID                                93
Unknown for age analysis; median-imputed for modeling only      7
Name: count, dtype: int64

In [26]:
df["age_group"].value_counts(dropna=False)

age_group
55-64      214
45-54      200
35-44      190
25-34      150
65+        146
18-24       93
Unknown      7
Name: count, dtype: int64

The original age column preserves the raw missing values. Instead of deleting rows with missing age, the dataset uses a transparent age-treatment approach. The age_cleaned column recovers missing ages using the same Customer ID when possible. Remaining unresolved ages are kept as Unknown in age_group for demographic analysis, while age_model is used only for predictive modeling after median imputation.

In [27]:
treatment_plan = pd.DataFrame({
    "Column": ["age", "age_cleaned", "age_group", "age_model", "region", "shipping_status"],
    "Issue": [
        "Original column has missing values",
        "Some ages remain unresolved after Customer ID recovery",
        "Contains Unknown category for unresolved ages",
        "Prepared numeric age column for modeling",
        "Contains Unknown values",
        "Contains Unknown values"
    ],
    "Treatment Decision": [
        "Preserve original values for transparency",
        "Use for EDA; keep remaining missing values unresolved",
        "Use Unknown category for demographic charts",
        "Use median-imputed age only for predictive modeling",
        "Keep Unknown as a valid category; do not impute",
        "Keep Unknown as a valid category; do not impute"
    ],
    "Reason": [
        "The original column documents the raw data condition",
        "Avoids inventing age values for demographic analysis",
        "Allows records to stay in the dataset without fake ages",
        "Predictive models require complete numeric values",
        "Region cannot be reliably inferred from other fields",
        "Shipping status cannot be reliably inferred from other fields"
    ]
})

treatment_plan

,Column,Issue,Treatment Decision,Reason
0,age,Original column has missing values,Preserve original values for transparency,The original column documents the raw data con...
1,age_cleaned,Some ages remain unresolved after Customer ID ...,Use for EDA; keep remaining missing values unr...,Avoids inventing age values for demographic an...
2,age_group,Contains Unknown category for unresolved ages,Use Unknown category for demographic charts,Allows records to stay in the dataset without ...
3,age_model,Prepared numeric age column for modeling,Use median-imputed age only for predictive mod...,Predictive models require complete numeric values
4,region,Contains Unknown values,Keep Unknown as a valid category; do not impute,Region cannot be reliably inferred from other ...
5,shipping_status,Contains Unknown values,Keep Unknown as a valid category; do not impute,Shipping status cannot be reliably inferred fr...
